# Polynomial Regression: Adam vs SGD + LR Scheduling
**Task ID:** `linreg_lvl6_adam_poly`  
**Series:** Linear Regression, Level 6  
**Protocol:** `pytorch_task_v1`

## Math
- Polynomial feature expansion (degree 3): $\phi(x) = [x, x^2, x^3]$
- Linear model on expanded features: $\hat{y} = w_0 + w_1 x + w_2 x^2 + w_3 x^3$
- MSE loss: $L = \frac{1}{N} \sum_i (\hat{y}_i - y_i)^2$
- Two optimizers compared:
  - **SGD** with momentum
  - **Adam** with CosineAnnealingLR scheduler

In [ ]:
import sys
import os
import json
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")

## Required Functions (`pytorch_task_v1` protocol)

In [ ]:
def get_task_metadata():
    return {
        "task_id": "linreg_lvl6_adam_poly",
        "algorithm": "Polynomial Regression (Adam vs SGD + LR Scheduling)",
        "series": "Linear Regression",
        "level": 6,
        "interface_protocol": "pytorch_task_v1",
    }

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

print(f"Device: {get_device()}")
print(f"Task: {get_task_metadata()['task_id']}")

In [ ]:
def poly_features(x, degree=3):
    """Expand 1-D tensor x into [x, x^2, ..., x^degree]."""
    return torch.stack([x ** d for d in range(1, degree + 1)], dim=1)

def make_dataloaders(cfg):
    n = cfg["n_samples"]
    degree = cfg["degree"]
    noise_std = cfg["noise_std"]
    batch_size = cfg["batch_size"]
    val_ratio = cfg.get("val_ratio", 0.2)

    x = torch.linspace(-math.pi, math.pi, n)
    y = torch.sin(x) + noise_std * torch.randn(n)

    X = poly_features(x, degree)

    # Standardize polynomial features (train stats only)
    n_val = int(n * val_ratio)
    idx = torch.randperm(n)
    train_idx, val_idx = idx[n_val:], idx[:n_val]

    X_mean = X[train_idx].mean(dim=0)
    X_std = X[train_idx].std(dim=0)
    X_norm = (X - X_mean) / (X_std + 1e-8)

    X_train, y_train = X_norm[train_idx], y[train_idx]
    X_val, y_val = X_norm[val_idx], y[val_idx]

    train_ds = TensorDataset(X_train, y_train.unsqueeze(1))
    val_ds = TensorDataset(X_val, y_val.unsqueeze(1))

    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True),
        DataLoader(val_ds, batch_size=batch_size),
        {"X_mean": X_mean, "X_std": X_std, "x_raw": x, "y_raw": y},
    )

In [ ]:
def build_model(cfg):
    return nn.Linear(cfg["degree"], 1)

def _train_one(model, train_loader, val_loader, epochs, optimizer, device, scheduler=None):
    """Train one model variant, return (train_losses, val_losses)."""
    criterion = nn.MSELoss()
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(X_b)
        train_losses.append(epoch_loss / len(train_loader.dataset))

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                val_loss += criterion(model(X_b), y_b).item() * len(X_b)
        val_losses.append(val_loss / len(val_loader.dataset))

        if scheduler:
            scheduler.step()

        if (epoch + 1) % 100 == 0:
            print(f"  Epoch {epoch+1}/{epochs} | Train MSE: {train_losses[-1]:.5f} | Val MSE: {val_losses[-1]:.5f}")

    return train_losses, val_losses

def train(model_sgd, model_adam, train_loader, val_loader, cfg, device):
    epochs = cfg["epochs"]

    print("--- Training SGD ---")
    opt_sgd = optim.SGD(model_sgd.parameters(), lr=cfg["sgd_lr"], momentum=cfg["sgd_momentum"])
    sgd_train, sgd_val = _train_one(model_sgd, train_loader, val_loader, epochs, opt_sgd, device)

    print("--- Training Adam + CosineAnnealingLR ---")
    opt_adam = optim.Adam(model_adam.parameters(), lr=cfg["adam_lr"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt_adam, T_max=epochs)
    adam_train, adam_val = _train_one(model_adam, train_loader, val_loader, epochs, opt_adam, device, scheduler)

    return {
        "sgd_train_losses": sgd_train, "sgd_val_losses": sgd_val,
        "adam_train_losses": adam_train, "adam_val_losses": adam_val,
    }

In [ ]:
def evaluate(model, dataloader, device):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for X_b, y_b in dataloader:
            X_b = X_b.to(device)
            all_preds.append(model(X_b).cpu())
            all_targets.append(y_b)
    preds = torch.cat(all_preds).squeeze()
    targets = torch.cat(all_targets).squeeze()
    mse = torch.mean((preds - targets) ** 2).item()
    ss_res = torch.sum((targets - preds) ** 2).item()
    ss_tot = torch.sum((targets - targets.mean()) ** 2).item()
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0
    return {"mse": mse, "rmse": math.sqrt(mse), "r2": r2}

def predict(model, X, device):
    model.eval()
    with torch.no_grad():
        return model(X.to(device)).cpu()

def save_artifacts(outputs, cfg):
    out_dir = cfg.get("output_dir", "./output")
    os.makedirs(out_dir, exist_ok=True)
    saveable = {k: v for k, v in outputs.items() if isinstance(v, (int, float, str, dict, list))}
    with open(os.path.join(out_dir, "metrics.json"), "w") as f:
        json.dump(saveable, f, indent=2, default=str)
    print(f"Metrics saved to {out_dir}")

## Configuration

In [ ]:
cfg = {
    "n_samples": 500,
    "degree": 3,
    "noise_std": 0.1,
    "batch_size": 64,
    "val_ratio": 0.2,
    "epochs": 500,
    "sgd_lr": 0.001,
    "sgd_momentum": 0.9,
    "adam_lr": 0.01,
    "mse_threshold": 0.1,
    "convergence_threshold": 0.05,
    "output_dir": "./output/linreg_lvl6_adam_poly",
}

## Train Both Models

In [ ]:
set_seed(42)
device = get_device()

train_loader, val_loader, data_info = make_dataloaders(cfg)

model_sgd = build_model(cfg).to(device)
model_adam = build_model(cfg).to(device)

history = train(model_sgd, model_adam, train_loader, val_loader, cfg, device)

## Evaluate

In [ ]:
sgd_metrics = evaluate(model_sgd, val_loader, device)
adam_metrics = evaluate(model_adam, val_loader, device)

print(f"SGD  - Val MSE: {sgd_metrics['mse']:.5f}  R2: {sgd_metrics['r2']:.4f}")
print(f"Adam - Val MSE: {adam_metrics['mse']:.5f}  R2: {adam_metrics['r2']:.4f}")

# Find convergence epochs
def find_convergence_epoch(losses, threshold):
    for i, l in enumerate(losses):
        if l < threshold:
            return i + 1
    return None

sgd_conv = find_convergence_epoch(history["sgd_val_losses"], cfg["convergence_threshold"])
adam_conv = find_convergence_epoch(history["adam_val_losses"], cfg["convergence_threshold"])
print(f"\nConvergence epoch (val MSE < {cfg['convergence_threshold']}):")
print(f"  SGD : {sgd_conv}")
print(f"  Adam: {adam_conv}")

## Visualization: Loss Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history["sgd_val_losses"], label="SGD val", color="blue", linewidth=1.5)
ax.plot(history["adam_val_losses"], label="Adam+Cosine val", color="orange", linewidth=1.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("SGD vs Adam+CosineAnnealingLR - Validation Loss")
ax.legend()
ax.set_yscale("log")
fig.tight_layout()
os.makedirs(cfg["output_dir"], exist_ok=True)
fig.savefig(os.path.join(cfg["output_dir"], "linreg_lvl6_loss_comparison.png"), dpi=120)
plt.show()

## Visualization: Polynomial Fit

In [ ]:
x_raw = data_info["x_raw"]
y_raw = data_info["y_raw"]

X_full = poly_features(x_raw, cfg["degree"])
X_full_norm = (X_full - data_info["X_mean"]) / (data_info["X_std"] + 1e-8)

model_adam_cpu = model_adam.cpu()
preds_adam = predict(model_adam_cpu, X_full_norm, torch.device("cpu")).squeeze().numpy()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x_raw.numpy(), y_raw.numpy(), ".", alpha=0.3, label="Data", color="gray")
ax.plot(x_raw.numpy(), np.sin(x_raw.numpy()), label="True sin(x)", color="green", linewidth=2)
ax.plot(x_raw.numpy(), preds_adam, label=f"Adam Poly-{cfg['degree']} Fit", color="red", linewidth=2)
ax.legend()
ax.set_title(f"Degree-{cfg['degree']} Polynomial Fit (Adam model)")
fig.tight_layout()
fig.savefig(os.path.join(cfg["output_dir"], "linreg_lvl6_fit.png"), dpi=120)
plt.show()

## Save Artifacts & Quality Assertions

In [ ]:
outputs = {
    **history,
    "sgd_val_metrics": sgd_metrics,
    "adam_val_metrics": adam_metrics,
    "sgd_convergence_epoch": sgd_conv,
    "adam_convergence_epoch": adam_conv,
}
save_artifacts(outputs, cfg)

# Quality assertions
print("\n" + "=" * 60)
print("Quality Assertions...")
print("=" * 60)

assert sgd_metrics["mse"] < cfg["mse_threshold"], \
    f"SGD final val MSE {sgd_metrics['mse']:.5f} >= {cfg['mse_threshold']}"
print(f"PASS: SGD val MSE {sgd_metrics['mse']:.5f} < {cfg['mse_threshold']}")

assert adam_metrics["mse"] < cfg["mse_threshold"], \
    f"Adam final val MSE {adam_metrics['mse']:.5f} >= {cfg['mse_threshold']}"
print(f"PASS: Adam val MSE {adam_metrics['mse']:.5f} < {cfg['mse_threshold']}")

if sgd_conv is not None and adam_conv is not None:
    assert adam_conv <= sgd_conv, \
        f"Adam did not converge faster than SGD ({adam_conv} vs {sgd_conv})"
    print(f"PASS: Adam converged in {adam_conv} epochs vs SGD {sgd_conv} epochs")

print("\n=== ALL CHECKS PASSED ===")
exit_code = 0
print(f"sys.exit({exit_code})")